# 单卡GPU 进行 ChatGLM3-6B模型 LORA 高效微调
本 Cookbook 将带领开发者使用 `AdvertiseGen` 对 ChatGLM3-6B 数据集进行 lora微调，使其具备专业的广告生成能力。

**硬件需求**

显存：24GB及以上（推荐使用30系或A10等sm80架构以上的NVIDIA显卡进行尝试）

## 一、准备数据集
我们使用 AdvertiseGen 数据集来进行微调。从 [Google Drive](https://drive.google.com/file/d/13_vf0xRTQsyneRKdD1bZIr93vBGOczrk/view?usp=sharing) 或者 [Tsinghua Cloud](https://cloud.tsinghua.edu.cn/f/b3f119a008264b1cabd1/?dl=1) 下载处理好的 AdvertiseGen 数据集，将解压后的 AdvertiseGen 目录放到 `/root/autodl-tmp/data/` 下。

### 1.1 AdvertiseGen 数据格式转换

In [8]:
import json           # json：标准库，用于 JSON 格式的序列化（dumps）和反序列化（loads）
from typing import Union  # Union：类型注解，表示参数可以是多种类型之一
from pathlib import Path  # Path：面向对象的文件系统路径类，支持路径拼接和文件操作


def _resolve_path(path: Union[str, Path]) -> Path:
    """
    将输入路径解析为绝对路径。

    参数：
        path (Union[str, Path]): 输入路径，支持字符串或 Path 对象，支持 '~' 用户目录展开

    返回值：
        Path: 展开用户目录并解析为绝对路径的 Path 对象
    """
    # expanduser() 将 '~' 展开为用户主目录，resolve() 将相对路径转为绝对路径
    return Path(path).expanduser().resolve()


def _mkdir(dir_name: Union[str, Path]):
    """
    递归创建目录，若目录已存在则跳过。

    参数：
        dir_name (Union[str, Path]): 要创建的目录路径，支持字符串或 Path 对象
    """
    # 将输入目录路径解析为绝对 Path 对象，类型：Path
    dir_name = _resolve_path(dir_name)
    if not dir_name.is_dir():  # 判断目录是否已存在，is_dir() 返回类型：bool
        # parents=True 表示递归创建中间目录，exist_ok=False 表示若目录已存在则报错（此处已提前判断）
        dir_name.mkdir(parents=True, exist_ok=False)


def convert_adgen(data_dir: Union[str, Path], save_dir: Union[str, Path]):
    """
    将 AdvertiseGen 原始数据集转换为 ChatGLM3 多轮对话微调格式（JSONL）。

    原始格式：每行为 {"content": "商品描述关键词", "summary": "广告文案"}
    目标格式：每行为 {"conversations": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}

    参数：
        data_dir (Union[str, Path]): 原始 AdvertiseGen 数据集目录，包含 train.json 和 dev.json
        save_dir (Union[str, Path]): 转换后数据的保存目录，会自动创建对应的子目录结构
    """
    def _convert(in_file: Path, out_file: Path):
        """
        将单个原始数据文件转换并保存为目标格式。

        参数：
            in_file (Path): 输入文件的绝对路径（原始 AdvertiseGen JSONL 文件）
            out_file (Path): 输出文件的绝对路径（转换后的对话格式 JSONL 文件）
        """
        # 确保输出文件的父目录存在，若不存在则递归创建
        _mkdir(out_file.parent)
        with open(in_file, encoding='utf-8') as fin:          # 以 UTF-8 编码打开原始输入文件
            with open(out_file, 'wt', encoding='utf-8') as fout:  # 以文本写模式打开输出文件
                for line in fin:  # 逐行读取原始文件，每行为一条 JSON 记录，类型：str
                    dct = json.loads(line)  # 将 JSON 字符串解析为 Python 字典，类型：dict
                    # 构建 ChatGLM3 对话格式的样本字典
                    # 'content' 作为用户输入（商品关键词），'summary' 作为 assistant 回复（广告文案）
                    sample = {'conversations': [
                        {'role': 'user', 'content': dct['content']},        # 用户消息（商品描述关键词）
                        {'role': 'assistant', 'content': dct['summary']}    # 助手回复（广告文案）
                    ]}
                    # 将样本字典序列化为 JSON 字符串并写入文件
                    # ensure_ascii=False 保留中文字符（不转义为 \uXXXX），类型写入：str + '\n'
                    fout.write(json.dumps(sample, ensure_ascii=False) + '\n')

    # 将数据目录解析为绝对 Path 对象，类型：Path
    data_dir = _resolve_path(data_dir)
    # 将保存目录解析为绝对 Path 对象，类型：Path
    save_dir = _resolve_path(save_dir)

    train_file = data_dir / 'train.json'  # 拼接训练集文件路径，类型：Path
    if train_file.is_file():  # 检查训练集文件是否存在，is_file() 返回类型：bool
        # 计算相对于 data_dir 的相对路径，并拼接到 save_dir 下，保持目录结构一致
        out_file = save_dir / train_file.relative_to(data_dir)  # 输出文件路径，类型：Path
        _convert(train_file, out_file)  # 转换训练集文件

    dev_file = data_dir / 'dev.json'  # 拼接验证集文件路径，类型：Path
    if dev_file.is_file():  # 检查验证集文件是否存在
        # 计算相对路径并拼接到保存目录，类型：Path
        out_file = save_dir / dev_file.relative_to(data_dir)
        _convert(dev_file, out_file)  # 转换验证集文件


# ★ 修改点：data 文件夹已移至 /root/autodl-tmp/data/，使用绝对路径
# 原路径：convert_adgen('data/AdvertiseGen', 'data/AdvertiseGen_fix')
convert_adgen('/root/autodl-tmp/data/AdvertiseGen', '/root/autodl-tmp/data/AdvertiseGen_fix')

## 二、使用命令行开始微调
接着，我们仅需要将配置好的参数以命令行的形式传参给程序，就可以使用命令行进行高效微调。

### 2.1 使用 LoRA 方法启动微调训练

In [ ]:
import os   # 导入标准库 os，用于在 Python 中设置环境变量（Windows 不支持 bash 内联变量赋值）
import sys  # 导入标准库 sys，sys.executable 返回当前内核使用的 Python 解释器绝对路径
            # 用 {sys.executable} 替代 !python，确保子进程与内核使用同一虚拟环境，
            # 避免 !python 调用系统 Python 导致找不到 jieba 等库

# ── 环境变量配置（Windows 下必须通过 os.environ 设置，不能使用 bash 的 KEY=VAL 前缀语法）──
os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # 指定只使用第 0 号 GPU 进行训练
os.environ['NCCL_P2P_DISABLE'] = '1'          # 禁用 NCCL 点对点（P2P）通信，避免部分 GPU 架构的兼容问题
os.environ['NCCL_IB_DISABLE'] = '1'           # 禁用 NCCL InfiniBand 网络通信（无 IB 网卡的环境下必须禁用）

# ── 本地 ChatGLM3-6B 模型路径（HuggingFace Hub 缓存的快照目录）──
# 该路径由 HuggingFace 自动下载并缓存，model 文件夹已移至 /root/autodl-tmp/model/ 下
# snapshots/<hash>/ 即为包含 config.json、pytorch_model 等完整权重文件的目录
# ★ 修改点1：model 文件夹已移至 /root/autodl-tmp/model/，使用绝对路径
# 原路径：model/ChatGLM3/model/models--THUDM--chatglm3-6b/snapshots/e9e0406d062cdb887444fe5bd546833920abd4ac
MODEL_DIR = "/root/autodl-tmp/model/ChatGLM3/model/models--THUDM--chatglm3-6b/snapshots/e9e0406d062cdb887444fe5bd546833920abd4ac"

# 启动单卡 LoRA 微调训练，各参数说明如下：
# 4.1.FineTune_CallByCL_ChatGLM3.py ：微调训练入口脚本（支持全参数微调、LoRA、P-Tuning v2）
# /root/autodl-tmp/data/AdvertiseGen_fix ：训练数据目录，包含已转换为 ChatGLM3 对话格式的 JSON 文件
# MODEL_DIR             ：ChatGLM3-6B 预训练基础模型本地快照路径（已更新为 autodl-tmp 绝对路径）
# /root/autodl-tmp/configs/lora_PEFTFrame_train.yaml ：LoRA 微调超参数配置文件（output_dir 已更新）
# -u：强制 Python 以无缓冲（unbuffered）模式运行，使训练日志和 tqdm 进度条实时刷新到 Jupyter 输出格
# 若不加 -u，Python 子进程默认使用块缓冲（~8KB），导致所有输出堆积后才一次性显示
# ★ 修改点2：脚本名由 finetune_hf.py 改为实际文件名 4.1.FineTune_CallByCL_ChatGLM3.py
# ★ 修改点3：data 路径和 configs 路径均已更新为 autodl-tmp 绝对路径
!{sys.executable} -u 4.1.FineTune_CallByCL_ChatGLM3.py /root/autodl-tmp/data/AdvertiseGen_fix {MODEL_DIR} /root/autodl-tmp/configs/lora_PEFTFrame_train.yaml

/root/miniconda3/envs/glm3/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
Loading checkpoint shards: 100%|█████████████████| 7/7 [00:01<00:00,  5.90it/s]
trainable params: 3,899,392 || all params: 6,247,483,392 || trainable%: 0.06241540401681151
--> Model

--> model has 3.899392M params

Setting num_proc from 16 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 114599 examples [00:00, 1005215.82 examples/s]
Setting num_proc from 16 back to 1 for the validation split to disable multiprocessing a

## 三、使用微调后的模型进行推理
在完成微调任务之后，我们可以查看到 `/root/autodl-tmp/output` 文件夹下多了很多个`checkpoint-*`的文件夹，这些文件夹代表了训练的轮数。
我们选择最后一轮的微调权重，并使用inference进行导入。

### 3.1 加载微调检查点进行推理

In [22]:
import os    # 导入标准库 os，用于设置环境变量和遍历目录
import sys   # 导入标准库 sys，sys.executable 返回当前内核使用的 Python 解释器绝对路径
import json  # 导入标准库 json，用于解析 trainer_state.json 获取最优检查点路径

# ── 环境变量配置（Windows 下通过 os.environ 设置，与 Cell 6 保持一致）──
os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # 指定只使用第 0 号 GPU 进行推理
os.environ['NCCL_P2P_DISABLE'] = '1'          # 禁用 NCCL P2P 通信
os.environ['NCCL_IB_DISABLE'] = '1'           # 禁用 NCCL InfiniBand 通信


def get_best_checkpoint(output_dir: str) -> str:
    """
    自动从训练输出目录中定位最优 LoRA adapter 检查点路径。

    HuggingFace Trainer 将 trainer_state.json 写入每个 checkpoint-* 子目录内。
    策略：
      1. 扫描所有 checkpoint-* 子目录，按步数降序排列；
      2. 从步数最大（即最新）的检查点目录中读取 trainer_state.json，
         提取 best_model_checkpoint 字段（需训练时配置 load_best_model_at_end=true），
         该字段始终指向验证集指标（此处为 rouge-l）最优的检查点目录；
      3. 若 trainer_state.json 不存在或字段缺失，则直接返回步数最大的检查点目录。

    参数：
        output_dir (str): 训练输出根目录的绝对路径，即 yaml 中 output_dir 对应的路径

    返回值：
        str: 最优检查点目录的绝对路径；若目录下无任何检查点则抛出 FileNotFoundError
    """
    # ── 第一步：收集所有 checkpoint-* 子目录并按步数降序排列 ──
    # os.listdir 返回 output_dir 下所有条目名称的列表，类型：list[str]
    ckpt_dirs = [
        d for d in os.listdir(output_dir)                        # 遍历 output_dir 下所有条目名称
        if d.startswith("checkpoint-")                           # 筛选以 "checkpoint-" 开头的条目
        and os.path.isdir(os.path.join(output_dir, d))          # 确认为目录而非文件
    ]
    if not ckpt_dirs:                                            # 若列表为空，说明尚无检查点
        raise FileNotFoundError(
            f"[ERROR] 在 {output_dir} 下未找到任何 checkpoint-* 目录，请确认训练已完成。"
        )
    # 按 "checkpoint-{step}" 中的 step 数值降序排序，类型：list[str]
    ckpt_dirs.sort(key=lambda d: int(d.split("-")[-1]), reverse=True)

    # ── 第二步：从步数最大的检查点目录中读取 trainer_state.json ──
    # trainer_state.json 由 Trainer 保存到每个 checkpoint-* 目录内，
    # 其中的 best_model_checkpoint 字段记录截至该检查点时验证集指标最优的目录路径
    latest_ckpt_dir = os.path.join(output_dir, ckpt_dirs[0])    # 步数最大的检查点目录路径，类型：str
    state_file = os.path.join(latest_ckpt_dir, "trainer_state.json")  # 状态文件路径，类型：str

    if os.path.isfile(state_file):                               # 检查文件是否存在，返回类型：bool
        with open(state_file, "r", encoding="utf-8") as f:       # 以 UTF-8 编码打开状态文件
            state = json.load(f)                                  # 解析 JSON，返回类型：dict
        best_ckpt = state.get("best_model_checkpoint")           # 获取最优检查点路径字段，类型：str | None
        if best_ckpt and os.path.isdir(best_ckpt):               # 路径非空且目录实际存在
            print(f"[INFO] 从 trainer_state.json 加载最优检查点: {best_ckpt}")
            return best_ckpt                                      # 返回最优检查点的绝对路径，类型：str

    # ── 第三步：回退策略——直接返回步数最大的检查点目录 ──
    print(f"[WARN] trainer_state.json 不可用，回退到步数最大的检查点: {latest_ckpt_dir}")
    return latest_ckpt_dir                                       # 返回步数最大检查点的绝对路径，类型：str


# ── 自动定位最优检查点 ──
# OUTPUT_DIR 对应 lora_autodl-tmp.yaml 中 training_args.output_dir 的值
OUTPUT_DIR = "/root/autodl-tmp/output"       # 训练输出根目录，类型：str
CHECKPOINT = get_best_checkpoint(OUTPUT_DIR)  # 最优 LoRA adapter 检查点目录，类型：str

# 使用微调后的 LoRA 检查点进行单轮对话推理，各参数说明如下：
# 4.2.inference_CallByCL_ChatGLM3.py ：推理脚本，自动检测目录中是否含 adapter_config.json，
#                     若存在则以 PEFT 模式加载 LoRA adapter，否则加载标准模型
# CHECKPOINT        ：由 get_best_checkpoint() 自动定位的最优检查点目录，含 LoRA adapter 权重
# --prompt          ：输入提示文本，格式为 "属性#值*属性#值" 的商品关键词组合，
#                     模型将据此生成对应的广告文案
!{sys.executable} 4.2.inference_CallByCL_ChatGLM3.py {CHECKPOINT} --prompt "类型#裙*版型#显瘦*材质#网纱*风格#性感*裙型#百褶*裙下摆#压褶*裙长#连衣裙*裙衣门襟#拉链*裙衣门襟#套头*裙款式#拼接*裙款式#拉链*裙款式#木耳边*裙款式#抽褶*裙款式#不规则" --temperature 0.55 --top-p 0.9 --max-length 512

[INFO] 从 trainer_state.json 加载最优检查点: /root/autodl-tmp/output/checkpoint-1000
Loading checkpoint shards: 100%|█████████████████| 7/7 [00:01<00:00,  3.60it/s]
Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.
这款连衣裙是选用轻薄透气的网纱材质，在视觉上有着一定的显瘦效果。整体的设计是套头式，方便穿脱。裙身采用百褶的压褶设计，不规则的裙摆，修饰腿型。裙衣的袖口是抽褶设计，性感而优雅。裙衣的领口是拉链设计，方便穿脱。裙衣下摆是不规则的拼接设计，增添层次感。


## 四、总结
到此位置，我们就完成了使用单张 GPU Lora 来微调 ChatGLM3-6B 模型，使其能生产出更好的广告。
在本章节中，你将会学会：
+ 如何使用模型进行 Lora 微调
+ 微调数据集的准备和对齐
+ 使用微调的模型进行推理